# Frozen NeuroLM-B encoder → ZuCo sentiment

This notebook tests whether the official pretrained NeuroLM-B **EEG encoder** transfers to natural-reading sentiment. It uses no sentence text and does not instantiate GPT-2. The frozen representation is evaluated against split-local shuffled EEG and majority controls.

Use a **GPU** Colab runtime and run every cell in order. All downloads occur in Colab; the 2.38 GB official checkpoint, compact feature cache, and results persist in Google Drive and are never committed to GitHub.

In [ ]:
# 1) Fetch this codebase, install small Colab-only dependencies, and pin upstream NeuroLM.
from pathlib import Path
import importlib.metadata as package_metadata
import os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")
UPSTREAM_URL = "https://github.com/935963004/NeuroLM.git"
UPSTREAM_COMMIT = "0cda9876d8ce6ee07ed0c43eee5e9a6f5c24b177"
UPSTREAM_ROOT = Path("/content/NeuroLM")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "neurolm/requirements-colab.txt")])
run([sys.executable, "-c", "from huggingface_hub import is_offline_mode; import transformers; print('dependency import check passed')"])
loaded_hub = sys.modules.get("huggingface_hub")
if loaded_hub is not None and not hasattr(loaded_hub, "is_offline_mode"):
    raise RuntimeError("huggingface_hub was already imported before its upgrade. Restart the Colab runtime, then run from Cell 1.")
print("huggingface_hub:", package_metadata.version("huggingface_hub"))
print("transformers:", package_metadata.version("transformers"))

if not (UPSTREAM_ROOT / ".git").exists():
    UPSTREAM_ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init"], cwd=UPSTREAM_ROOT)
    run(["git", "remote", "add", "origin", UPSTREAM_URL], cwd=UPSTREAM_ROOT)
run(["git", "fetch", "--depth", "1", "origin", UPSTREAM_COMMIT], cwd=UPSTREAM_ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=UPSTREAM_ROOT)
os.chdir(PROJECT_ROOT / "neurolm")
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=Path.cwd())
print("Project and tests ready:", Path.cwd())
print("Official NeuroLM commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip())

In [ ]:
# 2) Mount Drive and edit only these paths if your thesis layout differs.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
DATA_ROOT = THESIS_ROOT / "Data"
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/neurolm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/neurolm"
RAW_DIR = DATA_ROOT / "zuco_og_raw"
LABELS_CSV = DATA_ROOT / "zuco_sentiment_labels_task1_fixed.csv"
CHECKPOINT_ROOT = CACHE_ROOT / "upstream_checkpoints"
FEATURE_CACHE = CACHE_ROOT / "frozen_features_v1"
RESULTS_DIR = RESULTS_ROOT / "frozen_probe_v1"

for path in (RAW_DIR, LABELS_CSV):
    if not path.exists():
        raise FileNotFoundError(f"Edit this cell; not found: {path}")
for path in (CHECKPOINT_ROOT, FEATURE_CACHE, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Raw EEG:", RAW_DIR)
print("Feature cache:", FEATURE_CACHE)
print("Results:", RESULTS_DIR)

In [ ]:
# 3) Verify ZuCo and save the explicit EGI → NeuroLM spatial assignment.
import json
import pandas as pd
from src.channels import build_mne_spatial_mapping, select_usable_mapping
from src.zuco_io import inspect_zuco

inspection = pd.DataFrame(inspect_zuco(RAW_DIR, LABELS_CSV))
display(inspection)
print("Subjects:", len(inspection))
print("Label-matched rows:", int(inspection.matched_labels.sum()))
print("Usable recordings:", int(inspection.usable_for_neurolm.sum()))

mapping_all, mapping = select_usable_mapping(build_mne_spatial_mapping())
mapping_all.to_csv(RESULTS_DIR / "spatial_mapping.csv", index=False)
mapping_report = {
    "assignments_total": len(mapping_all),
    "assignments_used": len(mapping),
    "assignments_excluded_over_30_deg": int((~mapping_all.use_for_encoder).sum()),
    "unique_neurolm_channels": int(mapping.neurolm_index.nunique()),
    "used_mean_angular_distance_deg": float(mapping.angular_distance_deg.mean()),
    "used_max_angular_distance_deg": float(mapping.angular_distance_deg.max()),
    "excluded_channels": mapping_all.loc[~mapping_all.use_for_encoder, ["zuco_channel", "neurolm_channel", "angular_distance_deg"]].to_dict(orient="records"),
}
(RESULTS_DIR / "spatial_mapping_diagnostics.json").write_text(json.dumps(mapping_report, indent=2))
display(mapping_all.sort_values("angular_distance_deg", ascending=False).head(12))
print(json.dumps(mapping_report, indent=2))

In [ ]:
# 4) Materialize the authors' NeuroLM-B checkpoint in Drive and run one smoke test.
# The official checkpoint is a PyTorch pickle. Load it only from the pinned authors' repository.
import torch
from huggingface_hub import hf_hub_download
from src.config import CHECKPOINT_FILENAME, CHECKPOINT_REPOSITORY, PreprocessConfig
from src.official_neurolm import OfficialNeuroLMEncoder
from src.preprocess import preprocess_eeg
from src.zuco_io import iter_zuco_recordings

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
CHECKPOINT_PATH = Path(hf_hub_download(
    repo_id=CHECKPOINT_REPOSITORY,
    filename=CHECKPOINT_FILENAME,
    local_dir=CHECKPOINT_ROOT,
))
checkpoint_bytes = CHECKPOINT_PATH.stat().st_size
print(f"Checkpoint: {CHECKPOINT_PATH} ({checkpoint_bytes / 1e9:.3f} GB)")
if checkpoint_bytes < 2_000_000_000:
    raise IOError("Checkpoint is unexpectedly small or incomplete")
provenance = {
    "checkpoint_repository": CHECKPOINT_REPOSITORY,
    "checkpoint_filename": CHECKPOINT_FILENAME,
    "checkpoint_bytes": checkpoint_bytes,
    "upstream_commit": UPSTREAM_COMMIT,
}
(RESULTS_DIR / "checkpoint_provenance.json").write_text(json.dumps(provenance, indent=2))

encoder = OfficialNeuroLMEncoder(
    UPSTREAM_ROOT, CHECKPOINT_PATH, mapping.neurolm_index.to_numpy(),
    zuco_indices=mapping.zuco_index.to_numpy(), device="cuda"
)
example = next(iter_zuco_recordings(RAW_DIR, LABELS_CSV))
example_eeg = preprocess_eeg(example.eeg, PreprocessConfig())
example_feature, example_details = encoder.encode_recording(example_eeg)
print("Subject/sentence/label:", example.subject, example.sentence_id, example.label)
print("Raw → prepared → feature:", example.eeg.shape, example_eeg.shape, example_feature.shape)
print("Encoder details:", example_details)
print("Checkpoint load:", encoder.load_report)

In [ ]:
# 5) Extract/resume every reader-sentence feature. Each completed row is saved immediately.
from src.extraction import extract_feature_cache

manifest = extract_feature_cache(
    raw_dir=RAW_DIR,
    labels_csv=LABELS_CSV,
    cache_dir=FEATURE_CACHE,
    encoder=encoder,
    preprocess_config=PreprocessConfig(),
    overwrite=False,
    progress_every=25,
)
print(manifest["report"] | {"failures": len(manifest["report"]["failures"])})
del encoder
torch.cuda.empty_cache()

In [ ]:
# 6) Aggregate readers, save diagnostics, and run the locked sentence-level evaluation.
from src.config import EvaluationConfig
from src.evaluation import bootstrap_alignment_delta, evaluate_features, viability_gate
from src.features import build_sentence_features

X, y, metadata, diagnostics = build_sentence_features(FEATURE_CACHE)
diagnostic_totals = {key: value for key, value in diagnostics.items() if key != "records"}
metadata.to_csv(RESULTS_DIR / "sentence_metadata.csv", index=False)
diagnostics["records"].to_csv(RESULTS_DIR / "feature_diagnostics_records.csv", index=False)
(RESULTS_DIR / "feature_diagnostics.json").write_text(json.dumps(diagnostic_totals, indent=2))
print(diagnostic_totals)
display(metadata.describe(include="all"))

evaluation_config = EvaluationConfig()
metrics, predictions, summary = evaluate_features(
    X, y, metadata.sentence_id.to_numpy(), RESULTS_DIR, evaluation_config
)
display(summary)
delta = bootstrap_alignment_delta(
    predictions,
    samples=evaluation_config.bootstrap_samples,
    confidence=evaluation_config.bootstrap_ci,
)
gate = viability_gate(metrics, delta, evaluation_config)
(RESULTS_DIR / "alignment_delta.json").write_text(json.dumps(delta, indent=2))
(RESULTS_DIR / "viability_gate.json").write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=2))
print("Saved results:", RESULTS_DIR)

## After the run

Record the mapping distances, extraction failures, Colab GPU, aligned/shuffled scores, each seed delta, bootstrap interval, and gate decision in `neurolm/PROJECT_LOG.md`. A failed gate is a valid domain-transfer result; do not tune it away. V2 is permitted only if V1 passes the locked gate.